## **Projeto:** Merca Data Platform
### **Squad:** 2 | Camada Silver — ecommerce_itens_pedido
### Origem e Destino
| Item | Valor |

| **Origem** | `squad2/bronze/ecommerce_itens_pedido` (Delta Lake) |

| **Destino** | `squad2/silver/ecommerce_itens_pedido` (Delta Lake) |

| **Checkpoint** | `squad2/control/silver/ecommerce_itens_pedido/control_file.json` |

| **Dependência** | Bronze das 3 tabelas com status `CONCLUIDO` |

| **Polling** | Verifica novos snapshots a cada 30 segundos |

### Regras de Qualidade Aplicadas
| # | Regra | Campo(s) | Ação |

| 1 | Schema completo | 6 colunas obrigatórias | Bloqueia lote inteiro |

| 2 | Integridade referencial | `id_pedido` | Linha vai para Sala de Espera |

| 3 | Quantidade positiva | `quantidade` | Linha descartada |

| 4 | Consistência de preço | `preco_unitario` | Linha vai para Quarentena |

| 5 | Deduplicação | `id_item_pedido` | Duplicata descartada |

### Colunas de Auditoria
| Coluna | Descrição |

| `silver_processed_at` | Timestamp de processamento na Silver |


In [0]:
%pip install deltalake

In [0]:
%run ../utils/feat_squad2_99_helpers

- Configuração de Caminhos
Define os caminhos ABFSS para todas as tabelas envolvidas no processamento:
- Origem (Bronze de itens)
- Destino (Silver de itens)
- Dependências (Silver/Bronze de pedidos e produtos)
- Áreas de governança (Sala de Espera e Quarentena)

In [0]:
import logging
import pandas as pd
from deltalake import DeltaTable, write_deltalake
from datetime import datetime

logging.getLogger("azure").setLevel(logging.WARNING)

TABELA = "ecommerce_itens_pedido"
CAMADA = "silver"

path_bronze = get_delta_path("bronze", TABELA)
path_silver = get_delta_path(CAMADA, TABELA)

inicio = log_inicio("feat_squad2_" + CAMADA + "_" + TABELA)
log.info("Tabela : " + TABELA)
log.info("Camada : " + CAMADA)
log.info("Path   : " + path_silver)

In [0]:
def processar_snapshot(source_ref: str) -> bool:
    try:
        log.info("Lendo Bronze filtrado: " + source_ref)

        dt_bronze   = DeltaTable(path_bronze, storage_options=get_storage_options())
        df_snapshot = dt_bronze.to_pandas(
            filters=[("bronze_source_file", "=", source_ref)]
        )

        if df_snapshot.empty:
            log.warning("Nenhum dado na Bronze para: " + source_ref)
            return False

        total = len(df_snapshot)
        log.info("Linhas lidas da Bronze: " + str(total))

        # ITP-R01 Completude: schema completo
        COLUNAS_OBRIGATORIAS = [
            "id_item_pedido", "id_pedido", "sku",
            "quantidade", "preco_unitario", "desconto_aplicado"
        ]
        colunas_faltantes = [c for c in COLUNAS_OBRIGATORIAS if c not in df_snapshot.columns]
        if colunas_faltantes:
            raise ValueError("BLOQUEIO ITP-R01: Colunas ausentes: " + str(colunas_faltantes))
        log.info("ITP-R01: Schema completo validado.")

        # ITP-R05 Unicidade interna no lote
        antes = len(df_snapshot)
        df_working = df_snapshot.drop_duplicates(subset=["id_item_pedido"], keep="last").copy()
        log.info("ITP-R05 lote: " + str(antes - len(df_working)) + " duplicata(s) removida(s).")

        # ITP-R05 Unicidade externa contra Silver existente
        if DeltaTable.is_deltatable(path_silver, storage_options=get_storage_options()):
            df_silver_atual = DeltaTable(
                path_silver, storage_options=get_storage_options()
            ).to_pandas(columns=["id_item_pedido", "bronze_source_file"])

            sources_gravados = df_silver_atual["bronze_source_file"].unique().tolist()
            if source_ref in sources_gravados:
                log.info("Source ja gravado na Silver - ignorando: " + source_ref)
                return True

            ids_existentes = set(df_silver_atual["id_item_pedido"].unique())
            antes = len(df_working)
            df_working = df_working[~df_working["id_item_pedido"].isin(ids_existentes)].copy()
            log.info("ITP-R05 ext: " + str(antes - len(df_working)) + " id(s) ja existentes removidos.")

        # ITP-R03 Dominio: quantidade maior que 0
        antes = len(df_working)
        df_working = df_working[df_working["quantidade"] > 0].copy()
        log.info("ITP-R03: " + str(antes - len(df_working)) + " descartado(s) por quantidade invalida.")

        # ITP-R04 Dominio: preco_unitario maior que 0
        antes = len(df_working)
        df_working = df_working[df_working["preco_unitario"] > 0].copy()
        log.info("ITP-R04: " + str(antes - len(df_working)) + " descartado(s) por preco invalido.")

        # ITP-R07 Consistencia: quantidade inteiro
        antes = len(df_working)
        df_working = df_working[
            df_working["quantidade"] == df_working["quantidade"].astype(int)
        ].copy()
        log.info("ITP-R07: " + str(antes - len(df_working)) + " descartado(s) por quantidade decimal.")

        if df_working.empty:
            log.warning("Nenhuma linha valida para gravar na Silver.")
            return True

        df_working["silver_processed_at"] = datetime.now()

        for col in df_working.columns:
            if pd.api.types.is_datetime64_any_dtype(df_working[col]):
                df_working[col] = df_working[col].dt.tz_localize(None)

        write_deltalake(
            table_or_uri    = path_silver,
            data            = df_working,
            mode            = "append",
            storage_options = get_storage_options()
        )

        log.info("Gravado na Silver: " + str(len(df_working)) + " linhas.")
        return True

    except Exception as e:
        log.error("Erro ao processar " + source_ref + ": " + str(e))
        return False

### Validação Pontual
Execute esta célula isoladamente para verificar o estado atual sem iniciar o loop contínuo.

In [0]:
try:
    dt_bronze           = DeltaTable(path_bronze, storage_options=get_storage_options())
    df_bronze           = dt_bronze.to_pandas(columns=["bronze_source_file"])
    sources_disponiveis = sorted(df_bronze["bronze_source_file"].dropna().unique().tolist())
    processados         = ler_checkpoint(CAMADA, TABELA)
    novos               = [s for s in sources_disponiveis if s not in processados]

    log.info("Sources disponiveis : " + str(len(sources_disponiveis)))
    log.info("Ja processados      : " + str(len(processados)))
    log.info("Novos para processar: " + str(len(novos)))
    log.info("Status atual        : " + str(ler_status_checkpoint(CAMADA, TABELA)))

    dep_ok = camada_anterior_concluida(CAMADA, TABELAS_SQUAD2)
    log.info("Bronze CONCLUIDO    : " + str("Sim" if dep_ok else "Aguardando"))

    if not novos:
        log.info("Silver " + TABELA + " em dia!")
    else:
        for s in novos:
            print("Novo: " + s)

except Exception as e:
    log.error("Erro na validacao: " + str(e))
    raise

### Execucao Direta
Executada pelo Databricks Job apos Bronze concluida.

In [0]:
dt_bronze           = DeltaTable(path_bronze, storage_options=get_storage_options())
df_bronze           = dt_bronze.to_pandas(columns=["bronze_source_file"])
sources_disponiveis = sorted(df_bronze["bronze_source_file"].dropna().unique().tolist())
processados         = ler_checkpoint(CAMADA, TABELA)
novos               = [s for s in sources_disponiveis if s not in processados]

if not camada_anterior_concluida(CAMADA, TABELAS_SQUAD2):
    log.warning("Bronze nao concluida - encerrando Silver " + TABELA)
else:
    if not novos:
        log.info("Silver " + TABELA + " em dia.")
    else:
        log.info(str(len(novos)) + " source(s) novo(s) encontrado(s).")
        salvar_checkpoint(CAMADA, TABELA, processados, status="PROCESSANDO")

        for source_ref in novos:
            log.info("Processando: " + source_ref)
            sucesso = processar_snapshot(source_ref)
            if sucesso:
                processados.add(source_ref)
                log.info("OK: " + source_ref)
            else:
                log.warning("FALHOU: " + source_ref)

        salvar_checkpoint(CAMADA, TABELA, processados, status="CONCLUIDO")
        log.info("Silver " + TABELA + " concluida.")

log_fim("feat_squad2_" + CAMADA + "_" + TABELA, inicio)